### Genetic Algorithm

Proyek ini mengimplementasikan **Algoritma Genetika (Genetic Algorithm)** untuk menyelesaikan masalah pemilihan barang (*Knapsack Problem*). 

### Studi Kasus & Konsep Dasar:
* **Cerita:** Kita mau pergi *camping* membawa tas ransel berkapasitas maksimal **15 kg**. Ada 5 pilihan barang dengan berat dan nilai kegunaan (*value*) yang berbeda.
* **Tujuan GA:** Mencari kombinasi barang terbaik agar total nilai barang setinggi mungkin tanpa membuat tas melebihi kapasitas 15 kg.
* **Binary Encoding:** Solusi direpresentasikan sebagai string biner sepanjang $n$ bit (5 bit untuk 5 barang), di mana `1` berarti barang dibawa dan `0` berarti barang ditinggal.

In [1]:
import random
import matplotlib.pyplot as plt

print("test")

test


### Membuat data barang dan parameter awal

In [2]:
#Data Barang
items = [
    {"name": "Tenda", "weight": 7, "value": 50},
    {"name": "Sleeping Bag", "weight": 3, "value": 20},
    {"name": "Kompor Portable", "weight": 2, "value": 15},
    {"name": "Makanan", "weight": 5, "value": 30},
    {"name": "Kamera", "weight": 1, "value": 10}
]

max_weight = 15
chromo_length = len(items)

pop_size = 10
CR = 0.8 #Crossover rate
MR = 1 / chromo_length #Mutation Rate
generations = 30
elite_k = 1


### Membuat Kromosom Biner acak sepanjang jumlah barang

In [3]:
def create_chromosome():
    return [random.randint(0,1) for _ in range(chromo_length)]

### Membuat populasi awal secara acak


In [4]:
population = [create_chromosome() for _ in range(pop_size)]
print("Contoh populasi awal", population)

Contoh populasi awal [[0, 0, 0, 1, 0], [1, 1, 0, 0, 1], [1, 1, 1, 0, 1], [1, 1, 0, 0, 0], [0, 1, 1, 0, 0], [0, 0, 0, 0, 1], [1, 0, 0, 0, 1], [0, 1, 0, 0, 0], [0, 1, 0, 1, 0], [0, 0, 1, 1, 0]]


### Membuat fungsi yang menghitung fitness kromosom

In [5]:
def calculate_fitness(chromosome):
    total_weight = 0
    total_value = 0

    for i in range(len(chromosome)):
        if chromosome[i] == 1:
            total_weight += items[i]["weight"]
            total_value += items[i]["value"]
    
    if total_weight > max_weight:
        return 0

    return total_value

### Roulette wheel selection

In [22]:
def roulette_wheel_selection(population, fitnesses):
    total_fitness = sum(fitnesses)

    if total_fitness == 0:
        return random.choice(population).copy()

    cumulative_probs = []
    current_sum = 0
    for f in fitnesses:
        current_sum += (f / total_fitness)
        cumulative_probs.append(current_sum)

    r = random.random()
    for i,prob in enumerate(cumulative_probs):
        if r <= prob:
            return population[i].copy()
    return population[-1].copy()

### Crossover

In [7]:
def crossover(parent1, parent2):
    if random.random() < CR:
        cut = random.randint(1, chromo_length - 1)
        child1 = parent1[:cut] + parent2[cut:]
        child2 = parent2[:cut] + parent1[cut:]
        return child1, child2
    return parent1.copy(), parent2.copy()

### Mutation

In [8]:
def mutate(chromosome):
    mutated = chromosome.copy()
    for i in range(len(mutated)):
        if random.random() < MR:
            mutated[i] = 1 if mutated[i] == 0 else 0
    return mutated

### Generasi & Elitisme(Loop)

In [13]:
best_fitness_history = []
best_solution = None
best_fitness_ever = -1

for gen in range(generations):
    fitnesses = [calculate_fitness(ind) for ind in population]

    max_fit_idx = fitnesses.index(max(fitnesses))
    current_best_fit = fitnesses[max_fit_idx]
    current_best_ind = population[max_fit_idx]

    if current_best_fit > best_fitness_ever:
        best_fitness_ever = current_best_fit
        best_solution = current_best_ind

    best_fitness_history.append(current_best_fit)


    new_population = []
    new_population.append(current_best_ind.copy())

    while len(new_population) < pop_size:
        parent1 = roulette_wheel_selection(population, fitnesses)
        parent2 = roulette_wheel_selection(population, fitnesses)

        child1, child2 = crossover(parent1, parent2)

        child1 = mutate(child1)
        child2 = mutate(child2)

        new_population.append(child1)
        if len(new_population) < pop_size:
            new_population.append(child2)
    population = new_population


print("HASIL AKHIR")
print("Fitness Maksimal Ditemukan:", best_fitness_ever)
print("Kromosom Terbaik (Binary):", best_solution)


print("Barang yang dibawa dalam Ransel:")
total_w = 0
for i, val in enumerate(best_solution):
    if val == 1:
        print(f"- {items[i]['name']} (Berat: {items[i]['weight']} kg, Nilai: {items[i]['value']})")
        total_w += items[i]['weight']
print(f"Total Berat: {total_w} kg / {max_weight} kg")

HASIL AKHIR
Fitness Maksimal Ditemukan: 105
Kromosom Terbaik (Binary): [1, 0, 1, 1, 1]
Barang yang dibawa dalam Ransel:
- Tenda (Berat: 7 kg, Nilai: 50)
- Kompor Portable (Berat: 2 kg, Nilai: 15)
- Makanan (Berat: 5 kg, Nilai: 30)
- Kamera (Berat: 1 kg, Nilai: 10)
Total Berat: 15 kg / 15 kg
